# `StructureSurface` example: single protein (p53)

Walks through the full structure-aware FINCHES workflow on the **single-chain** p53
AlphaFold model `AF-P04637-F1-model_v6.pdb`:

1. Load + decompose into folded domains and IDRs.
2. Folded-domain surface vs buried.
3. The contiguous-surface "net".
4. Context-aware FINCHES scoring against an IDR.
5. Polymer-reach constraint on what an anchored IDR can touch.

In [1]:
from finches.frontend.mpipi_frontend import Mpipi_frontend
from finches.utils.structure_surface import StructureSurface
import numpy as np

# the frontend is passed once at construction; StructureSurface uses its IMC_object
# internally for all interaction scoring (so we never touch the IMC directly).
mf = Mpipi_frontend()
ss = StructureSurface("AF-P04637-F1-model_v6.pdb", mf)

/Users/alex/.uv/neuron_uv/.venv/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


## 1. Decomposition into folded domains and IDRs

In [2]:
print(f"chains       : {len({r.chain_index for r in ss.residues})}")
print(f"residues     : {len(ss.residues)}")
print(f"folded       : {len(ss.folded_residues)}")
print(f"IDR          : {len(ss.idr_residues)}")
print(f"IDR segments : {len(ss.idr_segments)}")
for seg in ss.idr_segments:
    first = seg["residues"][0].res_seq
    last  = seg["residues"][-1].res_seq
    n_anc = seg["n_anchor"].res_seq if seg["n_anchor"] else None
    c_anc = seg["c_anchor"].res_seq if seg["c_anchor"] else None
    print(f"  IDR {first:>4}-{last:<4} (len {len(seg['residues']):>3})  "
          f"N-anchor={n_anc}  C-anchor={c_anc}")

chains       : 1
residues     : 393
folded       : 179
IDR          : 214
IDR segments : 2
  IDR    1-103  (len 103)  N-anchor=None  C-anchor=104
  IDR  283-393  (len 111)  N-anchor=282  C-anchor=None


## 2. Folded-domain surface
SASA is computed on the folded domains only, so dangling IDR tails do not occlude the surface.

In [3]:
print(f"surface residues : {len(ss.surface_residues)} of {len(ss.folded_residues)} folded")

surface residues : 140 of 179 folded


## 3. Contiguous-surface net
Residues are linked only if their segment hugs the surface (KD-tree neighbours pruned by an interior-occlusion test).

In [4]:
g = ss.surface_graph
degrees = [d for _, d in g.degree()]
print(f"nodes / edges : {g.number_of_nodes()} / {g.number_of_edges()}")
print(f"mean degree   : {np.mean(degrees):.2f}")

# the patch of one surface residue (centre + its contiguous-surface neighbours)
key = ss.surface_residues[len(ss.surface_residues) // 2].key
patch = ss.surface_patch(key)
print(f"example patch around {key}: "
      f"{[ss.get_residue(*k).one_letter for k in patch]}")

nodes / edges : 140 / 208
mean degree   : 2.97
example patch around (0, 190): ['P', 'P', 'G']


## 4. FINCHES surface interaction scoring
Challenge the surface with a strongly acidic and a strongly basic IDR. A basic IDR should be most attractive at the surface's acidic residues.

In [5]:
acidic = "EEDEEDEEDEEDEEDEEDEE"
basic  = "RRKRRKRRKRRKRRKRRKRR"

res_acidic = ss.surface_vs_idr(acidic)
res_basic  = ss.surface_vs_idr(basic)

mean_acidic = np.mean([v["score"] for v in res_acidic.values()])
mean_basic  = np.mean([v["score"] for v in res_basic.values()])
print(f"mean score (acidic IDR): {mean_acidic:+.3f}")
print(f"mean score (basic  IDR): {mean_basic:+.3f}")

# most attractive surface residues for the basic IDR
top = sorted(res_basic.items(), key=lambda kv: kv[1]['score'])[:8]
print("\nmost attractive surface residues for the basic IDR:")
for k, info in top:
    print(f"  chain {k[0]} resSeq {k[1]:>4} ({info['one_letter']}): {info['score']:+.3f}")

mean score (acidic IDR): -0.136
mean score (basic  IDR): -0.087

most attractive surface residues for the basic IDR:
  chain 0 resSeq  186 (D): -2.629
  chain 0 resSeq  184 (D): -2.540
  chain 0 resSeq  198 (E): -2.447
  chain 0 resSeq  221 (E): -2.447
  chain 0 resSeq  224 (E): -2.447
  chain 0 resSeq  207 (D): -2.372
  chain 0 resSeq  228 (D): -2.372
  chain 0 resSeq  259 (D): -2.372


## 5. Polymer-reach constraint
`R(n) = 5 * n^0.54` Å. Anchor a basic IDR at a folded-domain junction and see what surface residues it can physically reach.

In [6]:
# largest IDR segment with a folded anchor
seg = max((s for s in ss.idr_segments if s['n_anchor'] or s['c_anchor']),
          key=lambda s: len(s['residues']))
anchor = (seg['c_anchor'] or seg['n_anchor']).key
idr_len = len(seg['residues'])

print(f"reach of residues 1, 10, 50, {idr_len}: "
      f"{np.round(ss.reach_radius([1, 10, 50, idr_len]), 1)}")

reachable = ss.reachable_surface_residues(anchor, idr_len)
print(f"IDR of length {idr_len} anchored at {anchor} reaches "
      f"{len(reachable)} / {len(ss.surface_residues)} surface residues")

# reach-gated scoring: distal residues see fewer IDR residues
gated  = ss.surface_vs_idr(basic, anchor=anchor, respect_reach=True)
nreach = np.array([v['n_reachable'] for v in gated.values()])
print(f"reach-gated contributing IDR residues: "
      f"min={nreach.min()} max={nreach.max()} (IDR length {len(basic)})")

reach of residues 1, 10, 50, 111: [ 5.  17.3 41.3 63.6]
IDR of length 111 anchored at (0, 282) reaches 140 / 140 surface residues
reach-gated contributing IDR residues: min=0 max=20 (IDR length 20)
